# validation tests


tests behind the methodology validation: theme coverage (tables 3 + 4), final-breakdown representativeness, golden-pair accuracy + positional bias (table 5), and regularization sensitivity.


In [ ]:
#imports
import os
import json
import time
import random
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from openai import OpenAI
import choix


In [ ]:
#configuration

#repo + paths
REPO_ROOT = os.path.abspath(
    os.path.join(os.getcwd(), "..")
    if os.path.basename(os.getcwd()) == "code"
    else os.getcwd()
)
BASE_DIR        = os.path.join(REPO_ROOT, "pairwise-comparison/output")
SAMPLE_DIR      = BASE_DIR
INPUT_DIR       = BASE_DIR
VALIDATION_DIR  = os.path.join(BASE_DIR, "validation_breakdowns")
BREAKDOWN_DIR   = os.path.join(BASE_DIR, "full_breakdowns")
RESULTS_DIR     = os.path.join(BASE_DIR, "comparison_results")
SCALING_DIR     = os.path.join(BASE_DIR, "scaling")
ANALYSIS_DIR    = os.path.join(BASE_DIR, "analysis")
GOLDEN_DIR      = os.path.join(BASE_DIR, "golden_candidates")
for d in (VALIDATION_DIR, ANALYSIS_DIR, GOLDEN_DIR):
    os.makedirs(d, exist_ok=True)

#llm settings (breakdowns + comparison)
SEED              = 42
MODEL             = "gpt-4o-mini"
TEMPERATURE       = 0
MAX_TOKENS        = 300
MAX_SPEECH_CHARS  = 4000
MAX_EXCERPT_CHARS = 1500
RATE_LIMIT_DELAY  = 0.3
MAX_RETRIES       = 3

#sampling thresholds
TARGET_PER_AREA    = 40
MIN_PER_CELL_PERIOD = 10
MIN_PER_CELL_YEAR   = 5

#periods + parties
PERIOD_BINS   = [2008, 2014, 2019, 2026]
PERIOD_LABELS = ["2009-2014", "2015-2019", "2020-2025"]

PARTY_IDEOLOGY = {
    "EL": "left", "SF": "left", "ALT": "left",
    "S": "centre-left", "RV": "centre-left",
    "V": "right", "KF": "right", "LA": "right",
    "DF": "right-national",
}
PARTY_COLORS = {
    "S":  "#E63946", "V":  "#2196F3", "DF":  "#FF9800",
    "SF": "#E91E63", "EL": "#B71C1C", "RV":  "#9C27B0",
    "KF": "#1565C0", "LA": "#00BCD4", "ALT": "#4CAF50",
    "NB": "#607D8B",
}

#final dimensions kept after theme validation (9 dim-keys across 6 areas)
KEEP_DIMS = {
    "labour_market_policy": ["conditional_welfare", "generosity_retrenchment", "welfare_chauvinism"],
    "education":            ["social_investment"],
    "pension":              ["fiscal_sustainability"],
    "elderly_care":         ["fiscal_sustainability"],
    "family":               ["social_investment", "generosity_retrenchment"],
    "housing":              ["welfare_chauvinism"],
}

#final scaling dimensions used for BT comparison
DIM_LABELS = {
    "labour_market_policy__welfare_retrenchment": "Labour Market: Welfare Retrenchment",
    "education__social_investment":               "Education: Social Investment",
    "pension__fiscal_sustainability":             "Pension: Fiscal Sustainability",
    "family__social_investment":                  "Family: Social Investment",
}
BREAKDOWN_FILE_MAP = {
    "labour_market_policy__welfare_retrenchment": "labour_market_policy_welfare_retrenchment_breakdowns.csv",
    "education__social_investment":               "education_social_investment_breakdowns.csv",
    "pension__fiscal_sustainability":             "pension_fiscal_sustainability_breakdowns.csv",
    "family__social_investment":                  "family_social_investment_breakdowns.csv",
}

random.seed(SEED)
np.random.seed(SEED)


In [ ]:
#area_dimensions breakdown prompts per area x dimension
AREA_DIMENSIONS = {

    #area: labour_market_policy (3 dimensions)
    "labour_market_policy": {

        "conditional_welfare": {
            "name": "Conditional vs Unconditional Welfare Rights",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on welfare conditionality in labour market policy.

DIMENSION: Conditional vs Unconditional Welfare Rights
- POLE A (Unconditional): Welfare as a right to income security during unemployment, independent of job search behaviour or activation requirements
- POLE B (Conditional): Welfare tied to labour market participation, activation measures, work obligations, and sanctions for non-compliance""",
            "user_prompt": """Analyze this speech for positioning on CONDITIONAL VS UNCONDITIONAL welfare rights.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether welfare should be conditional on work/activation or unconditional?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

NOTE: This dimension is about BEHAVIOURAL REQUIREMENTS — what the unemployed must DO to keep their benefits (job search, activation, accept work). It is NOT about the AMOUNT of benefits (that is generosity). It is NOT about WHO gets access based on nationality (that is a separate dimension).

What each pole sounds like:
- A Pole A speech defends the right to receive benefits without behavioural conditions — no mandatory activation, job search requirements, or sanctions. The focus is on whether support is unconditional, not on the benefit amount.
- A Pole B speech argues that receiving benefits comes with obligations — the speaker frames unemployment support as a contract where the state provides income in exchange for active job search, participation in programmes, or acceptance of offered work. The focus is on what the recipient must DO, not on how much they receive.

BREAKDOWN:""",
            "pole_a": "Unconditional income security during unemployment",
            "pole_b": "Work-first, activation, sanctions for non-compliance",
        },

        "generosity_retrenchment": {
            "name": "Generous vs Residual Benefit Levels",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on welfare benefit generosity in labour market policy.

DIMENSION: Generous vs Residual Benefit Levels
- POLE A (Generous): Unemployment benefits as adequate income replacement that maintains dignified living standards
- POLE B (Residual): Benefits deliberately capped or reduced to ensure work always pays more than welfare""",
            "user_prompt": """Analyze this speech for positioning on BENEFIT GENEROSITY VS RETRENCHMENT.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether benefit levels should be generous/adequate or capped/reduced?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

NOTE: This dimension is about the AMOUNT of benefits — whether levels are adequate or deliberately low. It is NOT about behavioural requirements for receiving benefits (that is conditionality). It is NOT about who qualifies based on nationality (that is a separate dimension).

What each pole sounds like:
- A Pole A speech argues that benefit levels are too low or must be protected to ensure dignified living standards. The focus is on the AMOUNT — replacement rates, adequacy, or the real purchasing power of benefits — not on the conditions attached to receiving them.
- A Pole B speech argues that the level of benefits is too high relative to wages, creating a financial disincentive to work. The focus is on the AMOUNT of the benefit — caps, ceilings, or reductions in replacement rates — not on whether recipients must fulfil behavioural requirements.

BREAKDOWN:""",
            "pole_a": "Adequate benefit levels for dignified living",
            "pole_b": "Capped/reduced benefits to incentivize work",
        },

        "welfare_chauvinism": {
            "name": "Universal vs Nationally Bounded Welfare Access",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on nationally bounded welfare rights in labour market policy.

DIMENSION: Universal vs Nationally Bounded Welfare Access
- POLE A (Universal): Welfare rights as entitlements based on legal residence, regardless of origin or integration status
- POLE B (Nationally bounded): Welfare access tied to citizenship, residency duration, or integration requirements""",
            "user_prompt": """Analyze this speech for positioning on UNIVERSAL VS NATIONALLY BOUNDED welfare access.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether welfare should be universally accessible to all residents or restricted based on nationality/integration?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

NOTE: This dimension is about WHO gets access based on nationality, origin, or residency duration. It is NOT about behavioural requirements that apply to everyone (that is conditionality). It is NOT about benefit levels or caps (that is generosity).

What each pole sounds like:
- A Pole A speech defends equal welfare access for all legal residents regardless of origin, citizenship status, or length of stay. The speaker may criticize differential treatment of immigrants as discriminatory or contrary to welfare state principles.
- A Pole B speech argues that welfare access should depend on contribution, residency duration, or integration into Danish society. The speaker frames restrictions as fair — newcomers should earn their way into the system before receiving full benefits.

BREAKDOWN:""",
            "pole_a": "Universal access based on residence",
            "pole_b": "Access tied to citizenship/integration requirements",
        },
    },

    #area: education (3 dimensions)
    "education": {

        "social_investment": {
            "name": "Education as Equality vs Economic Investment",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on the purpose of education.

DIMENSION: Education as Social Equality vs Economic Investment
- POLE A (Equality): Education as a universal right promoting social equality, democratic citizenship and personal formation
- POLE B (Investment): Education framed as investment in productivity, labour supply and the future workforce""",
            "user_prompt": """Analyze this speech for positioning on EDUCATION AS EQUALITY VS ECONOMIC INVESTMENT.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether education primarily serves social equality/personal formation or economic productivity/labour supply?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

NOTE: This dimension is about the PURPOSE of education — equality and personal formation vs economic productivity. It is NOT about behavioural requirements for receiving SU (that is conditionality). It is NOT about who should have access based on nationality (that is a separate dimension).

What each pole sounds like:
- A Pole A speech treats education as valuable in itself — for personal growth, democratic participation, or social mobility. The speaker resists framing education in terms of labour market outcomes or economic returns.
- A Pole B speech frames education as a means to strengthen the workforce and national competitiveness. The speaker evaluates education by its labour market relevance, graduate employment rates, or contribution to economic growth.

BREAKDOWN:""",
            "pole_a": "Education for equality, citizenship and personal formation",
            "pole_b": "Education as investment in productivity and labour supply",
        },

        "conditional_welfare": {
            "name": "Universal vs Conditional Education Benefits",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on behavioural conditionality in education benefits.

DIMENSION: Universal vs Conditional Education Benefits
- POLE A (Universal): SU as an unconditional right allowing students to study freely without performance pressure
- POLE B (Conditional): SU tied to progression speed, completion targets, or time limits""",
            "user_prompt": """Analyze this speech for positioning on UNIVERSAL VS CONDITIONAL education benefits.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether education benefits (SU) should be unconditional or tied to behavioural requirements like study speed or completion?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

NOTE: This dimension is about BEHAVIOURAL REQUIREMENTS — what students must DO to keep their benefits. It is NOT about WHO should have access based on nationality (that is a separate dimension).

What each pole sounds like:
- A Pole A speech defends student grants as a right that enables free and independent study without pressure to finish quickly or prove progression. The speaker treats SU as unconditional support for learning at one's own pace.
- A Pole B speech argues that student grants should come with expectations — faster completion, demonstrated progression, or time limits. The speaker frames open-ended study as wasteful and expects students to finish efficiently.

BREAKDOWN:""",
            "pole_a": "SU as unconditional right to study freely",
            "pole_b": "SU conditioned on progression speed and completion",
        },

        "welfare_chauvinism": {
            "name": "Universal vs Nationally Bounded Education Access",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on nationally bounded access to education benefits.

DIMENSION: Universal vs Nationally Bounded Education Access
- POLE A (Universal): Education and SU accessible to all students regardless of nationality or origin
- POLE B (Nationally bounded): Education benefits framed as a national investment that should primarily serve Danish students and taxpayers""",
            "user_prompt": """Analyze this speech for positioning on UNIVERSAL VS NATIONALLY BOUNDED access to education benefits.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether education benefits should be equally available to foreign students or reserved for Danish students/taxpayers?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

NOTE: This dimension is about WHO has access based on nationality or origin. It is NOT about behavioural requirements like study speed (that is a separate dimension).

What each pole sounds like:
- A Pole A speech defends equal access to education and SU for all students regardless of nationality, treating education as a universal right. The speaker may criticise restrictions on foreign students as discriminatory or counterproductive.
- A Pole B speech argues that Danish education and SU are investments by Danish taxpayers that should primarily benefit Denmark. The speaker may question why foreign students receive SU, argue for residency requirements, or frame unrestricted access as fiscally irresponsible.

BREAKDOWN:""",
            "pole_a": "Education benefits accessible to all regardless of nationality",
            "pole_b": "Education benefits reserved for Danish students/taxpayers",
        },
    },

    #area: pension (1 dimension)
    "pension": {

        "fiscal_sustainability": {
            "name": "Stable Social Contract vs Fiscal Sustainability",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on pension policy.

DIMENSION: Stable Social Contract vs Fiscal Sustainability
- POLE A (Social contract): Retirement as a fixed, earned right at a stable predictable age
- POLE B (Fiscal sustainability): Pension reforms justified by demographic pressure, including linking retirement age to life expectancy""",
            "user_prompt": """Analyze this speech for positioning on STABLE PENSION RIGHTS VS FISCAL SUSTAINABILITY.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether pensions should be a stable right or adjusted for demographic sustainability?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

What each pole sounds like:
- A Pole A speech defends a fixed retirement age as a promise that workers have earned through a lifetime of work. The speaker may emphasise physical wear, fairness to manual workers, or the right to a dignified retirement without further extensions.
- A Pole B speech argues that retirement age must adapt to rising life expectancy and demographic realities. The speaker frames later retirement as necessary for fiscal sustainability and intergenerational fairness.

BREAKDOWN:""",
            "pole_a": "Retirement as a stable earned right at a fixed age",
            "pole_b": "Pension age adjusted for demographics and fiscal sustainability",
        },
    },

    #area: health (1 dimension)
    "health": {

        "marketization": {
            "name": "Public Monopoly vs Mixed Public-Private Provision",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on healthcare provision.

DIMENSION: Public Monopoly vs Mixed Public-Private Provision
- POLE A (Public): The public healthcare system as the sole legitimate provider, where private insurance threatens solidarity
- POLE B (Mixed): Private providers and insurance complement the public system as a necessary supplement""",
            "user_prompt": """Analyze this speech for positioning on PUBLIC VS MIXED PUBLIC-PRIVATE healthcare.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether healthcare should be exclusively public or include private providers as a complement?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

What each pole sounds like:
- A Pole A speech argues that the public system must be the sole provider of healthcare, and that private alternatives create a two-tier system where access depends on wealth rather than need. Private insurance is seen as undermining solidarity.
- A Pole B speech accepts or promotes private healthcare alongside the public system — as a way to reduce waiting times, increase choice, or relieve pressure on public hospitals. The speaker frames public-private coexistence as pragmatic rather than threatening.

BREAKDOWN:""",
            "pole_a": "Universal public healthcare as sole provider",
            "pole_b": "Private insurance and providers complement public system",
        },
    },

    #area: elderly_care (2 dimensions)
    "elderly_care": {

        "marketization": {
            "name": "Public vs Mixed Provision in Elderly Care",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on elderly care provision.

DIMENSION: Public Monopoly vs Public-Private Complementarity in Elderly Care
- POLE A (Public): Elderly care as a universal public responsibility delivered through the municipal system
- POLE B (Mixed): Free choice of private care providers alongside the public system""",
            "user_prompt": """Analyze this speech for positioning on PUBLIC VS MIXED elderly care provision.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether elderly care should be exclusively public or include private providers?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

NOTE: This dimension is about WHO PROVIDES care — public vs private delivery. It is NOT about whether care budgets are fiscally sustainable (that is a separate dimension).

What each pole sounds like:
- A Pole A speech argues that elderly care is a public responsibility where private providers risk creating unequal treatment. The speaker defends municipal delivery as the guarantor of consistent, needs-based care.
- A Pole B speech promotes choice between public and private care providers as empowering for elderly citizens. The speaker frames competition or private alternatives as improving quality or efficiency.

BREAKDOWN:""",
            "pole_a": "Elderly care as exclusive public/municipal responsibility",
            "pole_b": "Free choice of private providers alongside public system",
        },

        "fiscal_sustainability": {
            "name": "Unconditional Care Right vs Fiscal/Demographic Pressure",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on elderly care financing.

DIMENSION: Unconditional Care Right vs Fiscal/Demographic Sustainability
- POLE A (Unconditional): Elderly care as an unconditional right that expands with need
- POLE B (Fiscal): Reforms justified by ageing population and fiscal sustainability of care budgets""",
            "user_prompt": """Analyze this speech for positioning on UNCONDITIONAL CARE VS FISCAL SUSTAINABILITY in elderly care.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether elderly care is an unconditional right or must adapt to fiscal realities?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

NOTE: This dimension is about whether care must ADAPT TO FISCAL CONSTRAINTS or expand unconditionally. It is NOT about whether care is delivered by public or private providers (that is a separate dimension).

What each pole sounds like:
- A Pole A speech argues that the state must provide care to match growing needs regardless of cost — demographic change is a political responsibility, not a reason for cutbacks. The speaker demands more resources and treats underfunding as a political choice.
- A Pole B speech argues that care provision must be realistic about fiscal constraints and demographic pressure. The speaker frames efficiency, prioritisation, or reform as necessary to maintain a sustainable system.

BREAKDOWN:""",
            "pole_a": "Elderly care as unconditional right expanding with need",
            "pole_b": "Care provision constrained by demographic and fiscal pressure",
        },
    },

    #area: family (4 dimensions)
    "family": {

        "conditional_welfare": {
            "name": "Universal vs Conditional Family Benefits",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on behavioural conditionality in family benefits.

DIMENSION: Universal vs Conditional Family Benefits
- POLE A (Universal): Child benefits as an unconditional universal right for all families
- POLE B (Conditional): Benefits tied to behavioural requirements such as daycare attendance or parental activity conditions""",
            "user_prompt": """Analyze this speech for positioning on UNIVERSAL VS CONDITIONAL family benefits.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether family benefits should be universal or conditional on parental behaviour?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

NOTE: This dimension is about BEHAVIOURAL REQUIREMENTS — what parents must DO to receive benefits (attend daycare, meet activity conditions, participate in programmes). It is NOT about the AMOUNT of benefits or caps (that is generosity). It is NOT about whether access differs by nationality or neighbourhood (that is a separate dimension).

What each pole sounds like:
- A Pole A speech defends family benefits as a universal right received without conditions on parental behaviour. The speaker resists mandatory participation requirements or activity conditions as undermining family autonomy.
- A Pole B speech argues that family benefits should require something in return — mandatory daycare enrolment, parental participation in programmes, or other behavioural expectations. The speaker frames conditions as ensuring benefits serve their intended purpose.

BREAKDOWN:""",
            "pole_a": "Universal family benefits without behavioural conditions",
            "pole_b": "Benefits conditional on parental behaviour or participation",
        },

        "welfare_chauvinism": {
            "name": "Universal vs Nationally Bounded Family Benefits",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on nationally bounded family benefits.

DIMENSION: Universal vs Nationally Bounded Family Benefits
- POLE A (Universal): Family benefits for all families with legal residence regardless of background
- POLE B (Nationally bounded): Benefits restricted or differentiated based on immigration status, ethnic background, or neighbourhood composition""",
            "user_prompt": """Analyze this speech for positioning on UNIVERSAL VS NATIONALLY BOUNDED family benefits.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether family benefits should apply equally to all residents or be differentiated based on origin, integration status, or neighbourhood?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

NOTE: This dimension is about WHO qualifies based on national origin, immigration status, or neighbourhood targeting. It is NOT about behavioural conditions that apply to all families equally (that is conditionality). It is NOT about benefit levels or caps (that is generosity).

What each pole sounds like:
- A Pole A speech defends equal family benefits for all residents regardless of ethnic background or neighbourhood. The speaker may criticise policies that single out immigrants or specific areas as discriminatory.
- A Pole B speech argues that family benefits for immigrants or residents of specific neighbourhoods should be restricted or come with additional requirements that do not apply to other families. The speaker frames differential treatment as necessary for integration or social cohesion.

BREAKDOWN:""",
            "pole_a": "Universal family benefits regardless of background",
            "pole_b": "Benefits differentiated by immigration status or neighbourhood",
        },

        "generosity_retrenchment": {
            "name": "Generous vs Residual Family Benefit Levels",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on family benefit generosity.

DIMENSION: Generous Universal vs Capped/Targeted Family Benefits
- POLE A (Generous): Family benefits at levels that genuinely support family life regardless of income
- POLE B (Residual): Benefits capped or means-tested to limit redistribution""",
            "user_prompt": """Analyze this speech for positioning on GENEROUS VS RESIDUAL family benefit levels.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether family benefit LEVELS should be generous for all or capped/reduced?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

NOTE: This dimension is about the AMOUNT of benefits — whether levels are adequate or should be capped/reduced. It is NOT about behavioural conditions for receiving benefits (that is conditionality). It is NOT about whether access differs by nationality (that is a separate dimension).

What each pole sounds like:
- A Pole A speech argues that family benefit levels must be maintained or increased to genuinely support family life. The speaker treats cuts or caps as an erosion of the welfare promise and defends universalism in the amount families receive.
- A Pole B speech argues that benefit levels should be reduced for high-income families or capped overall. The speaker frames means-testing of amounts as fiscally responsible and fairer than paying the same to wealthy and low-income families alike.

BREAKDOWN:""",
            "pole_a": "Universal family benefits at adequate levels for all",
            "pole_b": "Benefit levels capped or means-tested",
        },

        "social_investment": {
            "name": "Family Support vs Human Capital Investment",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on the purpose of family policy.

DIMENSION: Family Support vs Human Capital Investment
- POLE A (Family support): Family policy as support for family life and child well-being
- POLE B (Investment): Early childhood policies framed as investment in human capital and future labour supply""",
            "user_prompt": """Analyze this speech for positioning on FAMILY SUPPORT VS HUMAN CAPITAL INVESTMENT.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether family policy primarily serves family well-being or future economic productivity?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

What each pole sounds like:
- A Pole A speech frames childcare and family policy around the well-being of children and parents — quality time, child development, and supporting family life as intrinsically valuable.
- A Pole B speech frames early childhood policies as investments that pay off later — through better educational outcomes, higher labour force participation, or a stronger future tax base. Children are discussed as future workers or human capital.

BREAKDOWN:""",
            "pole_a": "Family policy supports family life and child well-being",
            "pole_b": "Family policy as investment in human capital and labour supply",
        },
    },

    #area: housing (2 dimensions)
    "housing": {

        "welfare_chauvinism": {
            "name": "Universal vs Nationally Bounded Housing Access",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on nationally bounded housing policy.

DIMENSION: Universal vs Nationally Bounded Housing Access
- POLE A (Universal): Social housing as universal infrastructure accessible to broad groups through social mixing
- POLE B (Nationally bounded): Housing policy focused on regulating specific neighbourhoods based on resident composition and integration concerns""",
            "user_prompt": """Analyze this speech for positioning on UNIVERSAL VS NATIONALLY BOUNDED housing access.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether social housing should be universally accessible or whether specific AREAS should be targeted based on their demographic composition?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

NOTE: This dimension is about AREA-LEVEL targeting — whether specific neighbourhoods should be regulated, transformed, or demolished based on who lives there. It is NOT about individual-level access requirements (that is a separate dimension).

What each pole sounds like:
- A Pole A speech defends social housing as mixed communities where different income groups and backgrounds live together. The speaker treats social housing as universal infrastructure and may criticise area-based targeting as stigmatising.
- A Pole B speech argues that certain neighbourhoods have become segregated and require special area-level intervention — changing resident composition, demolishing or transforming housing blocks, or designating specific areas for regulation based on the share of immigrants or non-Western residents.

BREAKDOWN:""",
            "pole_a": "Social housing as universal infrastructure for all groups",
            "pole_b": "Area-level targeting of neighbourhoods based on resident composition",
        },

        "conditional_welfare": {
            "name": "Universal vs Conditional Housing Access",
            "system_prompt": """You are analyzing Danish parliamentary speeches for ideological positioning on individual-level conditionality of housing access.

DIMENSION: Universal vs Conditional Access to Social Housing
- POLE A (Universal): Access based solely on individual housing need regardless of background
- POLE B (Conditional): Individual access tied to employment, residency duration, or integration requirements""",
            "user_prompt": """Analyze this speech for positioning on UNIVERSAL VS CONDITIONAL access to social housing.

SPEECH:
"{text}"

TASK:
Does this speech take a position on whether INDIVIDUAL access to social housing should be based on need alone or conditional on personal requirements?

If YES: Summarize the position in 2-3 sentences, noting key phrases.
If NO: Write only "NO POSITION ON THIS DIMENSION"

NOTE: This dimension is about INDIVIDUAL-LEVEL requirements — what a person must do or be to qualify for housing or housing support. It is NOT about area-level policies targeting specific neighbourhoods (that is a separate dimension).

What each pole sounds like:
- A Pole A speech argues that any person who needs housing should have equal access regardless of employment status, origin, or length of residency. The speaker treats housing as a basic right determined by need alone.
- A Pole B speech argues that individuals should meet personal conditions to access social housing or housing benefits — having a job, having lived in Denmark for a minimum period, or meeting integration requirements. The speaker frames individual conditions as ensuring resources go to those who contribute.

BREAKDOWN:""",
            "pole_a": "Housing access based solely on individual need",
            "pole_b": "Individual access conditioned on employment, residency, or integration",
        },
    },
}

#summary
_total = sum(len(d) for d in AREA_DIMENSIONS.values())
print(f"defined {_total} area x dimension combinations across {len(AREA_DIMENSIONS)} areas")
for area, dims in AREA_DIMENSIONS.items():
    print(f"  {area} ({len(dims)}): {', '.join(dims)}")


## theme validation - stratified sample


In [ ]:
#stratified sample (40 speeches per area, party x period) for prompt engineering

welfare_areas = list(AREA_DIMENSIONS.keys())
validation_samples = {}

for area in welfare_areas:
    df = pd.read_csv(os.path.join(SAMPLE_DIR, f"{area}_sample_final.csv"))
    df["period"] = pd.cut(df["year"], bins=PERIOD_BINS, labels=PERIOD_LABELS)

    cells_grid = df.groupby(["party", "period"], observed=True).size().reset_index(name="n")
    per_cell = max(1, TARGET_PER_AREA // max(len(cells_grid), 1))

    sampled = [g.sample(n=min(len(g), per_cell), random_state=SEED)
               for _, g in df.groupby(["party", "period"], observed=True)]
    df_val = pd.concat(sampled, ignore_index=True)

    if len(df_val) < TARGET_PER_AREA:
        remaining = df[~df["speech_id"].isin(df_val["speech_id"])]
        n_extra = min(TARGET_PER_AREA - len(df_val), len(remaining))
        df_val = pd.concat([df_val, remaining.sample(n=n_extra, random_state=SEED)], ignore_index=True)

    validation_samples[area] = df_val
    df_val.to_csv(os.path.join(SAMPLE_DIR, f"{area}_validation_sample.csv"), index=False)

    print(f"\n{area}: {len(df_val)} speeches")
    print(df_val.groupby(["party", "period"], observed=True).size().unstack(fill_value=0).to_string())

#cost estimate
total = sum(len(v) * len(AREA_DIMENSIONS[a]) for a, v in validation_samples.items())
print(f"\ntotal api calls planned: {total}  (~${total * 0.0003:.2f} at gpt-4o-mini)")


## theme validation - breakdown generation


In [ ]:
#openai client + breakdown helper

client = OpenAI()

def get_breakdown(system_prompt, user_prompt, speech_text):
    prompt = user_prompt.replace("{text}", speech_text[:MAX_SPEECH_CHARS])
    for attempt in range(MAX_RETRIES):
        try:
            r = client.chat.completions.create(
                model=MODEL, temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": prompt},
                ],
            )
            content = r.choices[0].message.content.strip()
            return {
                "breakdown": content,
                "has_position": "NO POSITION ON THIS DIMENSION" not in content.upper(),
                "tokens_used": r.usage.total_tokens,
                "status": "success",
            }
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                time.sleep((attempt + 1) * 5)
            else:
                return {"breakdown": f"ERROR: {e}", "has_position": False,
                        "tokens_used": 0, "status": "error"}


In [ ]:
#run breakdowns on the validation sample (resumable per area x dimension)

TEXT_COLS = ["text_clean", "text", "speech_text", "speech", "truncated_text"]

def _pick_text(row):
    for c in TEXT_COLS:
        if c in row.index and pd.notna(row[c]):
            return str(row[c])
    return None

def run_validation_breakdowns():
    total_calls = 0
    for area, dimensions in AREA_DIMENSIONS.items():
        val_path = os.path.join(SAMPLE_DIR, f"{area}_validation_sample.csv")
        if not os.path.exists(val_path):
            print(f"skipping {area}: no validation sample")
            continue

        df = pd.read_csv(val_path)
        print(f"\n{area}: {len(df)} speeches x {len(dimensions)} dims")

        for dim_key, cfg in dimensions.items():
            out_path = os.path.join(VALIDATION_DIR, f"{area}_{dim_key}_breakdowns.csv")
            existing = pd.read_csv(out_path) if os.path.exists(out_path) else None
            done = set(existing["speech_id"].values) if existing is not None else set()

            rows = []
            for _, row in df.iterrows():
                sid = row["speech_id"]
                if sid in done:
                    continue
                text = _pick_text(row)
                if not text or not text.strip():
                    continue
                res = get_breakdown(cfg["system_prompt"], cfg["user_prompt"], text)
                rows.append({
                    "speech_id": sid, "area": area, "dimension": dim_key,
                    "party": row.get("party", ""), "year": row.get("year", ""),
                    "speaker": row.get("speaker", ""),
                    "breakdown": res["breakdown"], "has_position": res["has_position"],
                    "tokens_used": res["tokens_used"], "status": res["status"],
                })
                total_calls += 1
                time.sleep(RATE_LIMIT_DELAY)

            if rows:
                new = pd.DataFrame(rows)
                combined = pd.concat([existing, new], ignore_index=True) if existing is not None else new
                combined.to_csv(out_path, index=False)
                print(f"  {dim_key}: saved {len(combined)} rows")

    print(f"\ntotal new calls: {total_calls}")

#guarded: this loop hits the openai api. enable when (re)running.
RUN_BREAKDOWN_GENERATION = False
if RUN_BREAKDOWN_GENERATION:
    run_validation_breakdowns()


## theme presence checks


In [ ]:
#load validation breakdowns

def load_breakdowns(folder):
    files = [f for f in os.listdir(folder)
             if f.endswith("_breakdowns.csv") and f != "all_validation_breakdowns.csv"]
    return pd.concat([pd.read_csv(os.path.join(folder, f)) for f in files], ignore_index=True)

df_val = load_breakdowns(VALIDATION_DIR)
print(f"validation breakdowns: {len(df_val)} rows across {df_val['area'].nunique()} areas")


In [ ]:
#position rate per area x dimension

def position_rates(df, title):
    print(title)
    print("-" * 60)
    for area in df["area"].unique():
        a = df[df["area"] == area]
        print(f"\n  {area}:")
        for dim in a["dimension"].unique():
            d = a[a["dimension"] == dim]
            pct = 100 * d["has_position"].mean() if len(d) else 0
            print(f"    {dim}: {d['has_position'].sum()}/{len(d)} ({pct:.0f}%)")

position_rates(df_val, "position rates - validation sample")


In [ ]:
#speeches with no position on any dimension in their area

print("speeches with no position on ANY dimension within their area")
print("-" * 60)
for area in df_val["area"].unique():
    a = df_val[df_val["area"] == area]
    any_pos = a.groupby("speech_id")["has_position"].any()
    n_none = (~any_pos).sum()
    pct = 100 * n_none / len(any_pos) if len(any_pos) else 0
    print(f"  {area}: {n_none}/{len(any_pos)} ({pct:.0f}%)")
    if n_none:
        details = a[a["speech_id"].isin(any_pos[~any_pos].index)][["speech_id", "party"]].drop_duplicates()
        print(f"    parties: {details['party'].value_counts().to_dict()}")


In [ ]:
#sanity checks on no-position vs has-position breakdowns

no_pos = df_val[~df_val["has_position"]]
print("unique no-position phrasings (first 80 chars)")
print(no_pos["breakdown"].str[:80].value_counts().to_string())

print("\nsample 'no position' breakdowns")
for (area, dim), g in no_pos.groupby(["area", "dimension"]):
    if len(g) == 0:
        continue
    print(f"\n  {area} / {dim} ({len(g)})")
    for _, r in g.head(2).iterrows():
        print(f"    {r['speech_id']} | {r['party']} | {r['year']}")
        print(f"    {r['breakdown'][:200]}")

print("\nsample 'has position' breakdowns (spot check)")
for area in ["education", "family", "housing"]:
    a = df_val[(df_val["area"] == area) & df_val["has_position"]]
    for dim in a["dimension"].unique():
        d = a[a["dimension"] == dim]
        if len(d) == 0:
            continue
        print(f"\n  {area} / {dim} ({len(d)})")
        for _, r in d.head(1).iterrows():
            print(f"    {r['speech_id']} | {r['party']}")
            print(f"    {r['breakdown'][:300]}")


## final breakdowns coverage & representativeness


In [ ]:
#load full breakdowns

df_all = load_breakdowns(BREAKDOWN_DIR)
df_all["period"] = pd.cut(df_all["year"], bins=PERIOD_BINS, labels=PERIOD_LABELS)
print(f"full breakdowns: {len(df_all)} rows | errors: {(df_all['status'] == 'error').sum()}")
position_rates(df_all, "\nposition rates - full sample")


In [ ]:
#within-area dimension overlap

overlap_rows, pairwise_rows = [], []

for area, a in df_all.groupby("area"):
    dims = list(a["dimension"].unique())
    if len(dims) < 2:
        continue
    sets = {d: set(a[(a["dimension"] == d) & a["has_position"]]["speech_id"]) for d in dims}
    any_dims = set.union(*sets.values()) if sets else set()
    all_dims = set.intersection(*sets.values()) if sets else set()

    for d in dims:
        others = set().union(*[s for k, s in sets.items() if k != d])
        unique = sets[d] - others
        overlap = sets[d] & others
        n = len(sets[d]) or 1
        overlap_rows.append({
            "area": area, "dimension": d,
            "total_with_position": len(sets[d]),
            "unique_to_dim": len(unique), "pct_unique": round(100 * len(unique) / n, 1),
            "overlap_with_other": len(overlap), "pct_overlap": round(100 * len(overlap) / n, 1),
        })
    overlap_rows.append({
        "area": area, "dimension": "-- area total --",
        "total_with_position": len(any_dims),
        "unique_to_dim": "-", "pct_unique": "-",
        "overlap_with_other": len(all_dims),
        "pct_overlap": round(100 * len(all_dims) / max(len(any_dims), 1), 1),
    })

    for da, db in combinations(dims, 2):
        both, either = sets[da] & sets[db], sets[da] | sets[db]
        pairwise_rows.append({
            "area": area, "dim_a": da, "dim_b": db,
            "position_in_both": len(both),
            "pct_of_either": round(100 * len(both) / max(len(either), 1), 1),
            "pct_of_dim_a":  round(100 * len(both) / max(len(sets[da]), 1), 1),
            "pct_of_dim_b":  round(100 * len(both) / max(len(sets[db]), 1), 1),
        })

print(pd.DataFrame(overlap_rows).to_string(index=False))
print()
print(pd.DataFrame(pairwise_rows).to_string(index=False))


In [ ]:
#representativeness: which parties survive per dimension

def representativeness(df, group_keys, min_per_cell, label):
    out = []
    for area, dims in KEEP_DIMS.items():
        for dim in dims:
            d = df[(df["area"] == area) & (df["dimension"] == dim) & df["has_position"]]
            counts = d.groupby(group_keys, observed=True).size().unstack(fill_value=0)
            keep, drop = [], []
            for party in counts.index:
                active = counts.loc[party][counts.loc[party] > 0]
                if len(active) < 2:
                    drop.append((party, "only 1 active unit"))
                elif group_keys[-1] == "period":
                    if active.min() < min_per_cell:
                        drop.append((party, f"{active.min()} in {active.idxmin()}"))
                    else:
                        keep.append(party)
                else:
                    n_below = (active < min_per_cell).sum()
                    if n_below > len(active) * 0.5:
                        drop.append((party, f"{n_below}/{len(active)} below {min_per_cell}"))
                    else:
                        keep.append(party)
            out.append({"area": area, "dimension": dim,
                        "total_usable": len(d), "parties_kept": len(keep),
                        "kept": keep, "dropped": [p for p, _ in drop]})

    print(f"\n{label}")
    print(f"{'area':<25} {'dimension':<28} {'usable':>7} {'kept':>5}  dropped")
    print("-" * 90)
    for s in out:
        print(f"{s['area']:<25} {s['dimension']:<28} {s['total_usable']:>7} "
              f"{s['parties_kept']:>5}  {', '.join(s['dropped']) or 'none'}")
    return out

_ = representativeness(df_all, ["party", "period"], MIN_PER_CELL_PERIOD,
                       f"period-level (min {MIN_PER_CELL_PERIOD} per cell)")
_ = representativeness(df_all, ["party", "year"], MIN_PER_CELL_YEAR,
                       f"year-level (min {MIN_PER_CELL_YEAR} per cell)")


In [ ]:
#capture rate: available -> sampled -> usable per dimension

for area, dims in KEEP_DIMS.items():
    speeches_path = os.path.join(SAMPLE_DIR, f"{area}_speeches.csv")
    n_available = len(pd.read_csv(speeches_path)) if os.path.exists(speeches_path) else None
    for dim in dims:
        sample = df_all[(df_all["area"] == area) & (df_all["dimension"] == dim)]
        usable = sample[sample["has_position"]]
        print(f"{area} / {dim}")
        if n_available:
            sr = 100 * len(sample) / n_available
            cr = 100 * len(usable) / n_available
            pr = 100 * len(usable) / max(len(sample), 1)
            print(f"  available={n_available}  sampled={len(sample)} ({sr:.0f}%)  "
                  f"usable={len(usable)} ({pr:.0f}% of sample, {cr:.0f}% of available)")
        else:
            print(f"  sampled={len(sample)}  usable={len(usable)} "
                  f"({100 * len(usable) / max(len(sample), 1):.0f}% of sample)")
        print()


## golden pair sampling


In [ ]:
#sample candidate golden pairs stratified by ideology x period

for area, dims in KEEP_DIMS.items():
    df_speeches = pd.read_csv(os.path.join(SAMPLE_DIR, f"{area}_sample_final.csv"))
    for dim in dims:
        br = pd.read_csv(os.path.join(BREAKDOWN_DIR, f"{area}_{dim}_breakdowns.csv"))
        br = br[br["has_position"]].copy()
        df = br.merge(df_speeches[["speech_id", "text_clean"]], on="speech_id", how="left")
        df["period"] = pd.cut(df["year"], bins=PERIOD_BINS, labels=PERIOD_LABELS)
        df["ideology_group"] = df["party"].map(PARTY_IDEOLOGY)

        sampled = []
        for (g, p), gd in df.groupby(["ideology_group", "period"], observed=True):
            uniq = gd.drop_duplicates(subset="speaker")
            n = min(2, len(uniq))
            if n:
                sampled.append(uniq.sample(n=n, random_state=SEED))
        if not sampled:
            continue
        ds = pd.concat(sampled, ignore_index=True).sort_values(["ideology_group", "year"])

        print(f"\n{area} / {dim} - {len(ds)} candidates from {len(df)} usable")
        print(ds.groupby(["ideology_group", "period"], observed=True).size().unstack(fill_value=0).to_string())
        ds[["speech_id", "party", "year", "speaker", "ideology_group",
            "period", "breakdown", "text_clean"]].to_csv(
            os.path.join(GOLDEN_DIR, f"{area}_{dim}.csv"), index=False)


## golden pair validation - run


In [ ]:
#comparison prompts + curated golden pairs (clear and hard cases) per scaling dimension

COMPARISON_PROMPTS = {
    "labour_market_policy__welfare_retrenchment": {
        "system_prompt": """You are comparing two Danish parliamentary speeches on the dimension of WELFARE RETRENCHMENT in labour market policy.

Your task is to determine which speech expresses a MORE RETRENCHMENT position — where welfare benefits are conditional on work behaviour AND/OR deliberately kept low to incentivize employment.

The opposite pole is UNCONDITIONAL GENEROUS WELFARE — where benefits are a right provided without behavioural conditions and at levels that maintain dignified living standards.""",
        "compare_direction": "more retrenchment-oriented (more conditional AND/OR more residual)",
    },
    "education__social_investment": {
        "system_prompt": """You are comparing two Danish parliamentary speeches on the dimension of EDUCATION AS EQUALITY VS ECONOMIC INVESTMENT.

Your task is to determine which speech expresses a MORE INVESTMENT-ORIENTED position — where education is framed as a means to strengthen the workforce and economic competitiveness.

The opposite pole is EQUALITY — where education is valued for personal formation, democratic citizenship, and social mobility.""",
        "compare_direction": "more investment-oriented",
    },
    "pension__fiscal_sustainability": {
        "system_prompt": """You are comparing two Danish parliamentary speeches on the dimension of STABLE PENSION RIGHTS VS FISCAL SUSTAINABILITY.

Your task is to determine which speech expresses a MORE FISCAL SUSTAINABILITY position — where pension age should adapt to demographics and life expectancy.

The opposite pole is STABLE SOCIAL CONTRACT — where retirement is a fixed earned right at a predictable age.""",
        "compare_direction": "more fiscal sustainability-oriented",
    },
    "elderly_care__fiscal_sustainability": {
        "system_prompt": """You are comparing two Danish parliamentary speeches on the dimension of UNCONDITIONAL CARE RIGHT VS FISCAL SUSTAINABILITY in elderly care.

Your task is to determine which speech expresses a MORE FISCAL SUSTAINABILITY position — where elderly care must adapt to demographic and budgetary constraints rather than expanding unconditionally.

The opposite pole is UNCONDITIONAL RIGHT — where care should expand to meet growing needs regardless of cost, and demographic pressure is a political responsibility not a justification for retrenchment.""",
        "compare_direction": "more fiscal sustainability-oriented",
    },
    "family__social_investment": {
        "system_prompt": """You are comparing two Danish parliamentary speeches on the dimension of FAMILY PROTECTION VS ECONOMIC INVESTMENT LOGIC.

Your task is to determine which speech expresses a MORE INVESTMENT-ORIENTED position — where family policy is justified by its economic returns, framing childcare and benefits around human capital, labour supply, and future productivity.

The opposite pole is FAMILY PROTECTION — where family policy is an unconditional social right supporting family life and child well-being regardless of economic returns.""",
        "compare_direction": "more investment-oriented",
    },
}

GOLDEN_PAIRS = {
    "education__social_investment": {
        "clear": [("20091_M27_helemoedet_t15",  "20091_M80_helemoedet_t565"),
                  ("20141_M70_helemoedet_t134", "20101_M19_helemoedet_t87"),
                  ("20151_M26_helemoedet_t15",  "20231_M81_helemoedet_t506")],
        "hard":  [("20091_M24_helemoedet_t206", "20091_M100_helemoedet_t1623"),
                  ("20171_M18_helemoedet_t11",  "20201_M138_helemoedet_t820")],
    },
    "family__social_investment": {
        "clear": [("20101_M64_helemoedet_t514", "20091_M16_helemoedet_t513"),
                  ("20141_M57_helemoedet_t303", "20131_M40_helemoedet_t62"),
                  ("20191_M65_helemoedet_t101", "20161_M6_helemoedet_t567")],
        "hard":  [("20091_M27_helemoedet_t93",  "20191_M32_helemoedet_t336"),
                  ("20141_M93_helemoedet_t697", "20191_M104_helemoedet_t263")],
    },
    "labour_market_policy__welfare_retrenchment": {
        "clear": [("20091_M102_helemoedet_t538", "20151_M107_helemoedet_t177"),
                  ("20101_M17_helemoedet_t103",  "20121_M35_helemoedet_t487"),
                  ("20091_M105_helemoedet_t136", "20191_M133_helemoedet_t115")],
        "hard":  [("20091_M102_helemoedet_t538", "20161_M102_helemoedet_t398")],
    },
    "pension__fiscal_sustainability": {
        "clear": [("20091_M100_helemoedet_t2048", "20101_M26_helemoedet_t409"),
                  ("20151_M17_helemoedet_t431",   "20111_M19_helemoedet_t395"),
                  ("20151_M90_helemoedet_t93",    "20222_M54_helemoedet_t25"),
                  ("20201_M22_helemoedet_t499",   "20111_M19_helemoedet_t395")],
        "hard":  [("20101_M103_helemoedet_t354",  "20101_M39_helemoedet_t173"),
                  ("20201_M27_helemoedet_t478",   "20201_M3_helemoedet_t602")],
    },
}

def load_speech_data(area, dim, speech_id):
    br = pd.read_csv(os.path.join(BREAKDOWN_DIR, f"{area}_{dim}_breakdowns.csv"))
    row = br[br["speech_id"] == speech_id]
    if row.empty:
        return None, None
    breakdown = row.iloc[0]["breakdown"]
    if "text_clean" in row.columns and pd.notna(row.iloc[0].get("text_clean")):
        return breakdown, str(row.iloc[0]["text_clean"])[:MAX_EXCERPT_CHARS]
    sp = pd.read_csv(os.path.join(SAMPLE_DIR, f"{area}_sample_final.csv"))
    tr = sp[sp["speech_id"] == speech_id]
    if tr.empty:
        return breakdown, None
    return breakdown, str(tr.iloc[0]["text_clean"])[:MAX_EXCERPT_CHARS]


In [ ]:
#run golden-pair comparisons (3 runs x both orders x both modes)

def _user_prompt(direction, breakdown_a, text_a, breakdown_b, text_b, include_text):
    if include_text:
        return f"""SPEECH A:
Summary: {breakdown_a}
Original excerpt: "{text_a}"

SPEECH B:
Summary: {breakdown_b}
Original excerpt: "{text_b}"

TASK: Which speech is {direction}?
Consider both the summary and the original language — pay attention to the strength of conviction, qualifications, and rhetorical framing, not just the direction.

ANSWER (one of the following):
- A (Speech A is {direction})
- B (Speech B is {direction})
- EQUAL (both express similar positions)

Provide your answer on the last line as: RESULT: [A/B/EQUAL]"""
    return f"""SPEECH A:
{breakdown_a}

SPEECH B:
{breakdown_b}

TASK: Which speech is {direction}?
Consider the strength of conviction, directness of the argument, and rhetorical framing — not just the direction of the position.

ANSWER (one of the following):
- A (Speech A is {direction})
- B (Speech B is {direction})
- EQUAL (both express similar positions)

Provide your answer on the last line as: RESULT: [A/B/EQUAL]"""

def _parse_result(content):
    for line in reversed(content.strip().split("\n")):
        s = line.strip().upper()
        if s.startswith("RESULT:"):
            r = s.replace("RESULT:", "").strip()
            if r in {"A", "B", "EQUAL"}:
                return r
    return "PARSE_ERROR"

def run_comparison(dim_key, sa, sb, flip=False, include_text=True):
    area, dim = dim_key.split("__")
    cfg = COMPARISON_PROMPTS[dim_key]
    ba, ta = load_speech_data(area, dim, sa)
    bb, tb = load_speech_data(area, dim, sb)
    if ba is None or bb is None:
        return {"result": "MISSING_DATA", "response": ""}
    if flip:
        ba, bb = bb, ba
        ta, tb = tb, ta
    user = _user_prompt(cfg["compare_direction"], ba, ta, bb, tb, include_text)
    r = client.chat.completions.create(
        model=MODEL, temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
        messages=[{"role": "system", "content": cfg["system_prompt"]},
                  {"role": "user",   "content": user}],
    )
    content = r.choices[0].message.content
    return {"result": _parse_result(content), "response": content}

def run_golden_pair_validation(n_runs=3):
    rows = []
    for run in range(n_runs):
        for dim_key, groups in GOLDEN_PAIRS.items():
            if dim_key not in COMPARISON_PROMPTS:
                continue
            for difficulty, pairs in groups.items():
                for pa, pb in pairs:
                    for include_text in (True, False):
                        for flip in (False, True):
                            out = run_comparison(dim_key, pa, pb, flip=flip, include_text=include_text)
                            raw = out["result"]
                            orig = ("B" if raw == "A" else "A") if flip and raw in {"A", "B"} else raw
                            correct = (orig == "B") or (orig == "EQUAL" and difficulty == "hard")
                            rows.append({
                                "run": run + 1, "dim_key": dim_key, "difficulty": difficulty,
                                "pole_a": pa, "pole_b": pb, "flipped": flip,
                                "mode": "with_text" if include_text else "breakdown_only",
                                "raw_result": raw, "original_result": orig,
                                "expected": "B", "correct": correct,
                            })
                            time.sleep(RATE_LIMIT_DELAY)
    return pd.DataFrame(rows)

#guarded: api-bound. flip when (re)running.
RUN_GOLDEN_PAIRS = False
if RUN_GOLDEN_PAIRS:
    results_df = run_golden_pair_validation(n_runs=3)
    results_df.to_csv(os.path.join(ANALYSIS_DIR, "golden_pair_results.csv"), index=False)
else:
    gp_path = os.path.join(ANALYSIS_DIR, "golden_pair_results.csv")
    results_df = pd.read_csv(gp_path) if os.path.exists(gp_path) else pd.DataFrame()
print(f"golden pair rows: {len(results_df)}")


In [ ]:
#summary: accuracy, positional bias, consistency

def golden_pair_summary(df):
    if df.empty:
        print("no golden pair results to summarize")
        return
    n_runs = df["run"].nunique()

    print(f"{'mode':<20} {'accuracy':>9} {'bias_A':>8} {'equal':>8} {'consistency':>12}")
    print("-" * 60)
    for mode in ["with_text", "breakdown_only"]:
        m = df[df["mode"] == mode]
        if m.empty:
            continue
        acc   = 100 * m["correct"].mean()
        bias  = 100 * (m["raw_result"] == "A").mean()
        eq    = 100 * (m["original_result"] == "EQUAL").mean()
        n_cons, n_pair = 0, 0
        for _, g in m.groupby(["dim_key", "pole_a", "pole_b", "flipped"]):
            if len(g) == n_runs:
                n_pair += 1
                if g["original_result"].nunique() == 1:
                    n_cons += 1
        cons = 100 * n_cons / n_pair if n_pair else 0
        print(f"{mode:<20} {acc:>8.0f}% {bias:>7.0f}% {eq:>7.0f}% {cons:>11.0f}%")

    print(f"\nper dimension")
    print(f"{'dimension':<45} {'text':>6} {'summ':>6} {'diff':>6}")
    print("-" * 65)
    for dk in df["dim_key"].unique():
        d = df[df["dim_key"] == dk]
        t = 100 * d[d["mode"] == "with_text"]["correct"].mean()
        s = 100 * d[d["mode"] == "breakdown_only"]["correct"].mean()
        print(f"{dk:<45} {t:>5.0f}% {s:>5.0f}% {t-s:>+5.0f}%")

    print(f"\nby difficulty")
    print(f"{'difficulty':<15} {'text':>6} {'summ':>6}")
    print("-" * 30)
    for diff in ["clear", "hard"]:
        d = df[df["difficulty"] == diff]
        if d.empty:
            continue
        t = 100 * d[d["mode"] == "with_text"]["correct"].mean()
        s = 100 * d[d["mode"] == "breakdown_only"]["correct"].mean()
        print(f"{diff:<15} {t:>5.0f}% {s:>5.0f}%")

    #pairs that are wrong in all modes
    print("\nflagged pairs (wrong in both modes)")
    for dk in df["dim_key"].unique():
        d = df[df["dim_key"] == dk]
        for (a, b), g in d.groupby(["pole_a", "pole_b"]):
            if g["correct"].sum() == 0:
                print(f"  {dk}: {a} vs {b}")
    print("\nbenchmark targets: accuracy > 80%  |  bias near 50%  |  consistency > 80%")

golden_pair_summary(results_df)


## regularization sensitivity & mle vs regularized


In [ ]:
#build comparisons + speech index for choix (uses one representative dimension for the sweep)

def build_comparisons(dim_key):
    df = pd.read_csv(os.path.join(RESULTS_DIR, f"{dim_key}_comparisons.csv"))
    df = df[df["result"].isin(["A", "B", "EQUAL"])].copy()
    speeches = sorted(set(df["speech_id_a"].tolist() + df["speech_id_b"].tolist()))
    idx = {s: i for i, s in enumerate(speeches)}
    pairs = []
    for _, row in df.iterrows():
        a, b = idx[row["speech_id_a"]], idx[row["speech_id_b"]]
        if   row["result"] == "A":     pairs.append((a, b))
        elif row["result"] == "B":     pairs.append((b, a))
        else:                          pairs.extend([(a, b), (b, a)])  #equal = half weight
    return df, speeches, idx, pairs

#metadata: speech-level party + year (from breakdown files)
def speech_meta(dim_key):
    br = pd.read_csv(os.path.join(BREAKDOWN_DIR, BREAKDOWN_FILE_MAP[dim_key]))
    return br[["speech_id", "party", "year"]].drop_duplicates()

#mle reference scores per party x year (output of notebook 02 scaling step)
mle_path = os.path.join(SCALING_DIR, "all_dimensions_year_scores_with_ci.csv")
df_mle_all = pd.read_csv(mle_path) if os.path.exists(mle_path) else pd.DataFrame()
print(f"comparison files: {sum(os.path.exists(os.path.join(RESULTS_DIR, f'{k}_comparisons.csv')) for k in DIM_LABELS)}/4 found")
print(f"mle reference scores: {'loaded' if not df_mle_all.empty else 'MISSING'} ({len(df_mle_all)} rows)")


In [ ]:
#alpha sweep on labour-market dimension (most data) -> pick alpha for all dims

SWEEP_DIM = "labour_market_policy__welfare_retrenchment"
ALPHAS    = [0.001, 0.01, 0.1, 1.0]

_, _, sidx_sw, pairs_sw = build_comparisons(SWEEP_DIM)
n_sw   = len(sidx_sw)
meta_sw = speech_meta(SWEEP_DIM)
mle_sw = df_mle_all[df_mle_all["dim_key"] == SWEEP_DIM] if not df_mle_all.empty else pd.DataFrame()

print(f"{'alpha':>7} {'r_vs_mle':>10} {'EL_range':>10} {'score_range':>22}")
print("-" * 55)
for a in ALPHAS:
    params = choix.opt_pairwise(n_sw, pairs_sw, alpha=a)
    s_map  = {s: params[i] for s, i in sidx_sw.items()}
    m = meta_sw.copy()
    m["bt_raw"] = m["speech_id"].map(s_map)
    if not mle_sw.empty:
        m["bt_std"] = (m["bt_raw"] - m["bt_raw"].mean()) / m["bt_raw"].std()
        py = m.groupby(["party", "year"])["bt_std"].mean().reset_index()
        merged = py.merge(mle_sw[["party", "year", "mean_score"]], on=["party", "year"])
        r, _ = stats.spearmanr(merged["bt_std"], merged["mean_score"]) if len(merged) else (float("nan"), 0)
    else:
        r = float("nan")
    el = m[m["party"] == "EL"]["bt_raw"]
    el_rng = (el.max() - el.min()) if len(el) else float("nan")
    print(f"{a:>7.3f} {r:>10.3f} {el_rng:>10.3f} "
          f"{params.min():>9.3f} to {params.max():.3f}")


In [ ]:
#compute regularized scores for all 4 final dimensions at chosen alpha

ALPHA_FINAL = 0.1

regularized_rows = []
for dim_key in DIM_LABELS:
    comp_path = os.path.join(RESULTS_DIR, f"{dim_key}_comparisons.csv")
    if not os.path.exists(comp_path):
        print(f"skip {dim_key}: no comparison file")
        continue
    _, _, sidx, pairs = build_comparisons(dim_key)
    params = choix.opt_pairwise(len(sidx), pairs, alpha=ALPHA_FINAL)
    s_map  = {s: params[i] for s, i in sidx.items()}
    m = speech_meta(dim_key).copy()
    m["bt_raw"] = m["speech_id"].map(s_map)
    m["bt_std"] = (m["bt_raw"] - m["bt_raw"].mean()) / m["bt_raw"].std()
    py = m.groupby(["party", "year"])["bt_std"].mean().reset_index()
    py = py.rename(columns={"bt_std": "mean_score"})
    py["dim_key"] = dim_key
    regularized_rows.append(py)

df_reg = pd.concat(regularized_rows, ignore_index=True) if regularized_rows else pd.DataFrame()
df_reg.to_csv(os.path.join(SCALING_DIR, f"all_dimensions_year_scores_regularized_alpha{ALPHA_FINAL}.csv"),
              index=False)
print(f"regularized scores: {len(df_reg)} party-year rows across {df_reg['dim_key'].nunique()} dims")


In [ ]:
#regularized vs mle correlation + floor effect inspection per dimension

print(f"{'dimension':<40} {'r':>7} {'p':>8} {'n':>5}  range_mle  range_reg")
print("-" * 80)
for dim_key, label in DIM_LABELS.items():
    reg = df_reg[df_reg["dim_key"] == dim_key]
    mle = df_mle_all[df_mle_all["dim_key"] == dim_key]
    if reg.empty or mle.empty:
        continue
    merged = reg.merge(mle[["party", "year", "mean_score"]].rename(columns={"mean_score": "mle_score"}),
                       on=["party", "year"])
    if len(merged) < 2:
        continue
    r, p = stats.spearmanr(merged["mean_score"], merged["mle_score"])
    mle_rng = f"{mle['mean_score'].min():.2f} to {mle['mean_score'].max():.2f}"
    reg_rng = f"{reg['mean_score'].min():.2f} to {reg['mean_score'].max():.2f}"
    print(f"{label:<40} {r:>7.3f} {p:>8.4f} {len(merged):>5}  {mle_rng:<10}  {reg_rng}")

#floor effect: focus on EL (most extreme party) per dimension
print(f"\nEL floor-effect inspection")
for dim_key, label in DIM_LABELS.items():
    reg = df_reg[(df_reg["dim_key"] == dim_key) & (df_reg["party"] == "EL")]
    mle = df_mle_all[(df_mle_all["dim_key"] == dim_key) & (df_mle_all["party"] == "EL")]
    if reg.empty or mle.empty:
        continue
    print(f"  {label}")
    print(f"    mle: range {mle['mean_score'].min():.2f} to {mle['mean_score'].max():.2f}, "
          f"{mle['mean_score'].nunique()} unique")
    print(f"    reg: range {reg['mean_score'].min():.2f} to {reg['mean_score'].max():.2f}, "
          f"{reg['mean_score'].nunique()} unique")


In [ ]:
#mle vs regularized score trajectories - 2x2 grid across the 4 final dimensions

def _plot_dim(ax, df, dim_key, title):
    sub = df[df["dim_key"] == dim_key]
    for party in sorted(sub["party"].unique()):
        p = sub[sub["party"] == party].sort_values("year")
        ax.plot(p["year"], p["mean_score"], marker="o", linewidth=1.8, markersize=3,
                label=party, color=PARTY_COLORS.get(party, "grey"))
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.4)
    ax.axvspan(2009, 2014, alpha=0.05, color="blue")
    ax.axvspan(2015, 2019, alpha=0.05, color="grey")
    ax.axvspan(2020, 2025, alpha=0.05, color="red")
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlim(2008.5, 2025.5)
    ax.set_xlabel("year")
    ax.set_ylabel("BT score (std)")
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
for i, (dim_key, label) in enumerate(DIM_LABELS.items()):
    _plot_dim(axes[i, 0], df_mle_all, dim_key, f"{label} — MLE")
    _plot_dim(axes[i, 1], df_reg,     dim_key, f"{label} — Regularized (alpha={ALPHA_FINAL})")
fig.suptitle("MLE vs Regularized Bradley-Terry trajectories", fontsize=12, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.985])
plt.savefig(os.path.join(ANALYSIS_DIR, "mle_vs_regularized_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.show()
